# Results Analysis

## Preparation

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import dotenv_values
from IPython.display import display
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env').is_file())
ENV = dotenv_values(REPO_ROOT / '.env')
DATA_DIR = Path(ENV['DATA_DIR']).expanduser()
FIGURES_DIR = REPO_ROOT / 'notebook/figures'
FIGURES_DIR.mkdir(exist_ok=True)
INPUT_DIR = DATA_DIR / 'analysis/sae-regression-seven-countries/inputs'
MODEL_ID = ENV.get('GENERATION_MODEL_SHORT', 'Meta-Llama-3.1-8B-Instruct')
JUDGE_ID = ENV.get('JUDGE_MODEL_ID', 'llama-3.1-70b-w8a8')
BASELINE_PATH = DATA_DIR / f'analysis/{JUDGE_ID}/output/default_bias/{MODEL_ID}.csv'
ABLATION_DIR = DATA_DIR / f'analysis-sae-off/{JUDGE_ID}/output/default_bias/{MODEL_ID}'
COUNTRIES = ['India', 'Japan', 'Korea, Republic of', 'Russian Federation']
ABLATION_COUNTRIES = [*COUNTRIES, 'Greece', 'Canada', 'Finland']
COUNTRY_LABELS = {'Finland': 'Finland',
                  'India': 'India', 'Japan': 'Japan', 'Korea, Republic of': 'Korea',
                  'Greece': 'Greece', 'Canada': 'Canada',
                  'Russian Federation': 'Russia'}
COUNTRY_COLORS = {'Finland': '#56B4E9',
                  'India': '#0072B2', 'Japan': '#D55E00',
                  'Korea, Republic of': '#009E73', 'Russian Federation': '#CC79A7',
                  'Greece': '#A6761D', 'Canada': '#666666'}
FACETS = {'Food': 'food', 'Clothing': 'clothing', 'Tradition': 'tradition', 'Rituals': 'rituals'}
EXTRA_ABLATION_FACETS = {'Greece': list(FACETS.values()), 'Canada': ['food', 'rituals'],
                         'Finland': ['clothing', 'tradition', 'rituals']}
TARGETS = ['Global', *FACETS]
LAYERS = [4, 8, 12, 16, 20, 24, 28]
CONFIGS = [f'L{layer}' for layer in LAYERS] + ['Multilayer']
RANDOM_STATE = 42
ALPHAS = np.logspace(-2, 3, 15)

sns.set_theme(style='whitegrid', context='paper', rc={'figure.dpi': 120, 'axes.spines.top': False,
                                                  'axes.spines.right': False, 'grid.alpha': .2})

In [ ]:
def relative_delta(after, before):
    return 100 * (after - before) / before.where(before.ne(0))


def with_average(frame):
    row = frame.select_dtypes('number').mean().to_dict()
    row['country'] = 'average'
    if 'facet' in frame:
        row['facet'] = None
    return pd.concat([frame, pd.DataFrame([row])], ignore_index=True)


def show_table(frame, rates=(), deltas=(), delta_unit='%'):
    formats = {column: '{:.2f} %' for column in rates}
    formats.update({column: f'{{:+.2f}} {delta_unit}' for column in deltas})
    styled = frame.style.hide(axis='index').format(formats, na_rep='NaN')
    if 'facet' in frame:
        styled = styled.format(na_rep='null', subset=['facet'])
    display(styled)


def judge_counts(path):
    if not path.is_file():
        return (np.nan, np.nan)
    decisions = pd.read_csv(path, sep='\t', usecols=['decision']).decision.str.strip().str.lower()
    return (decisions.eq('yes').sum(), len(decisions))


def load_bias(path):
    source = pd.read_csv(path).set_index('country')
    rows = [
        {'country': country, 'facet': target.lower(),
         'bias_name': source.loc[country, f'{target} With Name'],
         'bias_noname': source.loc[country, f'{target} Without Name'],
         'bias': source.loc[country, target]}
        for country in COUNTRIES for target in [*FACETS, 'Global']
    ]
    config = path.stem.replace('_top20_8x', '_8x_top20')
    judge_root = DATA_DIR / ('analysis' if path == BASELINE_PATH else 'analysis-sae-off') / JUDGE_ID
    for country in EXTRA_ABLATION_FACETS:
        counts = {}
        for target, facet in FACETS.items():
            counts[facet] = {}
            for mode in ['with_name', 'woname']:
                if path == BASELINE_PATH:
                    stem = (f'{facet}_FirstPerson_{country}_{country}' if mode == 'with_name'
                            else f'{facet}_FirstPerson_baseline_{country}')
                else:
                    slug = country.lower()
                    stem = f'{MODEL_ID}_{facet}_FirstPerson_{slug}_{mode}_{config}_sae_off_{slug}_{slug}'
                counts[facet][mode] = judge_counts(judge_root / mode / MODEL_ID / f'{stem}.tsv')
            with_yes, with_total = counts[facet]['with_name']
            without_yes, without_total = counts[facet]['woname']
            with_rate, without_rate = with_yes / with_total, without_yes / without_total
            rows.append({'country': country, 'facet': facet, 'bias_name': with_rate,
                         'bias_noname': without_rate, 'bias': with_rate - without_rate})
        included = EXTRA_ABLATION_FACETS[country]
        rates = {}
        for mode, column in [('with_name', 'bias_name'), ('woname', 'bias_noname')]:
            yes = sum(counts[facet][mode][0] for facet in included)
            total = sum(counts[facet][mode][1] for facet in included)
            rates[column] = yes / total
        rows.append({'country': country, 'facet': 'global', **rates,
                     'bias': rates['bias_name'] - rates['bias_noname']})
    return pd.DataFrame(rows)


def finish_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def prediction_plot(ax, observed, predicted, countries, title):
    for country in pd.unique(countries):
        positions = np.asarray(countries) == country
        ax.scatter(predicted[positions], observed[positions], s=25,
                   color=COUNTRY_COLORS[country], label=COUNTRY_LABELS[country])
    limits = [min(observed.min(), predicted.min()) - .02,
              max(observed.max(), predicted.max()) + .02]
    ax.plot(limits, limits, 'k--', linewidth=.8, alpha=.5)
    ax.set(xlabel='Predicted bias (OOF)', ylabel='Observed bias', title=title, xlim=limits, ylim=limits)
    ax.set_aspect('equal', adjustable='box')
    ax.legend(frameon=False, fontsize=8)


class FixedEffectsRidge(RegressorMixin, BaseEstimator):
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        self.n_features_in_ = X.shape[1]
        index = X[:, :1]
        controls = np.column_stack([np.ones(len(X)), X[:, 1:]])
        residual_index = index - controls @ np.linalg.lstsq(controls, index, rcond=None)[0]
        residual_y = y - controls @ np.linalg.lstsq(controls, y, rcond=None)[0]
        index_fit = Ridge(alpha=self.alpha, fit_intercept=False).fit(residual_index, residual_y)
        control_coef = np.linalg.lstsq(controls, y - index @ index_fit.coef_, rcond=None)[0]
        self.coef_ = np.concatenate([index_fit.coef_, control_coef[1:]])
        self.intercept_ = control_coef[0]
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_


def fit_model(estimator, X, y, countries):
    fitted = clone(estimator)
    if isinstance(fitted, GridSearchCV):
        fitted.cv = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
                         .split(X, countries))
    return fitted.fit(X, y)


def oof_prediction(estimator, X, y, splits, countries):
    summed, counts = np.zeros(len(y)), np.zeros(len(y))
    for train, test in splits:
        fitted = fit_model(estimator, X[train], y[train], countries[train])
        summed[test] += fitted.predict(X[test])
        counts[test] += 1
    return summed / counts


def regression_metrics(observed, predicted):
    return {'r2_oof': r2_score(observed, predicted), 'mae_oof': mean_absolute_error(observed, predicted)}

def benchmark_summary(folder, pattern, setting):
    if ENV.get(setting):
        return Path(ENV[setting]).expanduser()
    paths = sorted((DATA_DIR / folder).glob(pattern))
    if len(paths) != 1:
        raise ValueError(f'Expected one {folder} summary, found {len(paths)}. Set {setting} in .env.')
    return paths[0]


## SAE feature ablations

In [ ]:
baseline = load_bias(BASELINE_PATH)
ablation_summaries = {f'L{layer}': load_bias(ABLATION_DIR / f'L{layer}R_8x.csv') for layer in LAYERS}
multilayer_path = next(ABLATION_DIR.glob('multilayer_per_layer_*_top20_8x.csv'))
ablation_summaries['Multilayer'] = load_bias(multilayer_path)

mitigation = baseline.merge(ablation_summaries['Multilayer'], on=['country', 'facet'], suffixes=('', '_saeoff'))
mitigation = mitigation.loc[mitigation.facet.ne('global')].copy()
mitigation['delta_name'] = relative_delta(mitigation.bias_name_saeoff, mitigation.bias_name)
mitigation['delta_noname'] = relative_delta(mitigation.bias_noname_saeoff, mitigation.bias_noname)
bias_columns = ['bias_name', 'bias_noname', 'bias_name_saeoff', 'bias_noname_saeoff']
mitigation = mitigation[['country', 'facet', *bias_columns, 'delta_name', 'delta_noname']]
mitigation[bias_columns] *= 100
paired = mitigation.dropna(subset=['bias_name_saeoff', 'bias_noname_saeoff'])
mitigation_by_country = paired.groupby('country', sort=False).mean(numeric_only=True).reset_index()
mitigation_table = pd.concat([mitigation, with_average(paired).tail(1)], ignore_index=True)
mitigation_country_table = with_average(mitigation_by_country)
show_table(mitigation_table, bias_columns, ['delta_name', 'delta_noname'])

In [ ]:
show_table(mitigation_country_table, bias_columns, ['delta_name', 'delta_noname'])

In [ ]:
ablation_results = pd.concat([
    baseline.merge(frame, on=['country', 'facet'], suffixes=('_before', '_after')).assign(config=config)
    for config, frame in ablation_summaries.items()
], ignore_index=True)
ablation_results['reduction_pct'] = -relative_delta(ablation_results.bias_after.abs(), ablation_results.bias_before.abs())
global_rows = ablation_results.loc[ablation_results.facet.eq('global')]
mean_global = global_rows.groupby('config').reduction_pct.mean().reindex(CONFIGS)

fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(CONFIGS))
ax.bar(x, mean_global, color='#CBD5DA', width=.72, label='Country mean')
for offset, marker, country in zip(np.linspace(-.3, .3, len(ABLATION_COUNTRIES)),
                                   ['o', 's', '^', 'D', 'P', 'X', 'v'], ABLATION_COUNTRIES):
    values = global_rows.loc[global_rows.country.eq(country)].set_index('config').loc[CONFIGS, 'reduction_pct']
    ax.scatter(x + offset, values, marker=marker, s=30, color=COUNTRY_COLORS[country],
               edgecolor='white', linewidth=.4, label=COUNTRY_LABELS[country], zorder=3)
ax.axhline(0, color='black', linewidth=.8)
ax.axvline(len(LAYERS) - .5, color='#CCCCCC', linewidth=.8)
ax.set(xticks=x, xticklabels=CONFIGS, xlabel='SAE configuration', ylabel='Bias magnitude reduction (%)')
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.18), ncol=4, frameon=False)
finish_figure(fig, 'ablation_relative_reduction')

In [ ]:
facet_order = list(FACETS.values())
heatmap = ablation_results.groupby(['config', 'facet']).reduction_pct.mean().unstack().reindex(
    index=CONFIGS, columns=facet_order)
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heatmap, annot=True, annot_kws={'fontsize': 13}, fmt='.1f', cmap='BrBG', center=0, vmin=-100, vmax=100,
            linewidths=.5, cbar_kws={'label': 'Bias magnitude reduction (%)'}, ax=ax)
ax.set(xlabel='Facet', ylabel='SAE configuration')
ax.tick_params(axis='y', rotation=0)
finish_figure(fig, 'ablation_mean_heatmap')

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(10, 14), layout='constrained')
for ax, country in zip(axes.flat, ABLATION_COUNTRIES):
    matrix = ablation_results.loc[ablation_results.country.eq(country)].pivot(
        index='config', columns='facet', values='reduction_pct').reindex(index=CONFIGS, columns=facet_order)
    sns.heatmap(matrix, annot=True, annot_kws={'fontsize': 13}, fmt='.1f', cmap='BrBG', center=0, vmin=-100, vmax=100,
                linewidths=.5, cbar=False, ax=ax)
    ax.set(title=COUNTRY_LABELS[country], xlabel='', ylabel='')
    ax.tick_params(axis='y', rotation=0)
axes.flat[-1].set_axis_off()
colorbar = fig.colorbar(axes[0, 0].collections[0], ax=axes.flat[:len(ABLATION_COUNTRIES)].tolist(), shrink=.8)
colorbar.set_label('Bias magnitude reduction (%)')
fig.savefig(FIGURES_DIR / 'ablation_country_heatmaps.pdf', bbox_inches='tight')
plt.show()
plt.close(fig)

## Regressions

In [ ]:
names = pd.read_csv(INPUT_DIR / 'aligned_names.csv')
feature_map = pd.read_csv(INPUT_DIR / 'feature_map_all.csv')
contexts = pd.read_csv(INPUT_DIR / 'context_feature_provenance.csv')
with np.load(INPUT_DIR / 'name_activation_summaries.npz', allow_pickle=False) as saved:
    activations = {key: saved[key].copy() for key in ['freq_global', 'topk_mean']}
selected = (contexts.loc[contexts['rank'].le(20), ['layer', 'feature_id']].drop_duplicates()
            .merge(feature_map, on=['layer', 'feature_id']).sort_values('vector_index').reset_index(drop=True))
indices = selected.vector_index.to_numpy(dtype=int)
X_freq = activations['freq_global'][:, indices]
X_topk = activations['topk_mean'][:, indices]
index_features = contexts.loc[contexts['rank'].le(20)].merge(feature_map, on=['layer', 'feature_id'])
samples, model_matrices, encoders = {}, {}, {}
for target in TARGETS:
    matched = np.full(len(names), np.nan)
    for country in ABLATION_COUNTRIES:
        selection = index_features.loc[index_features.country.eq(country)]
        if target != 'Global':
            selection = selection.loc[selection.facet.eq(FACETS[target])]
        if selection.empty:
            continue
        positions = np.flatnonzero(names.country.eq(country))
        own_features = selection.vector_index.unique().astype(int)
        matched[positions] = activations['freq_global'][np.ix_(positions, own_features)].mean(axis=1)
    positions = np.flatnonzero(np.isfinite(matched))
    sample = names.iloc[positions].reset_index(drop=True)
    samples[target] = sample
    encoder = OneHotEncoder(drop='first', sparse_output=False)
    country = encoder.fit_transform(sample[['country']])
    encoders[target] = encoder
    freq, topk, index = X_freq[positions], X_topk[positions], matched[positions, None]
    model_matrices[target] = {
        'country': country,
        'country + freq': np.hstack([freq, country]),
        'country + topk_mean': np.hstack([topk, country]),
        'index': index,
        'country + index': np.hstack([index, country]),
        'freq': freq,
        'topk_mean': topk,
        'country + freq + topk_mean': np.hstack([freq, topk, country]),
    }
display(names.groupby('country').agg(n_names=('name', 'size'), n_docs=('n_docs', 'sum')))

In [ ]:
model_specs = {}
for model in model_matrices['Global']:
    model_specs[(model, 'OLS')] = make_pipeline(StandardScaler(), LinearRegression())
    ridge = FixedEffectsRidge() if model == 'country + index' else Ridge()
    parameter = f'{type(ridge).__name__.lower()}__alpha'
    model_specs[(model, 'Ridge')] = GridSearchCV(make_pipeline(StandardScaler(), ridge),
                                               {parameter: ALPHAS}, cv=5,
                                               scoring='neg_mean_squared_error', n_jobs=1)
rows, predictions = [], {}
for target in TARGETS:
    sample = samples[target]
    countries, y = sample.country.to_numpy(), sample[target].to_numpy()
    splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
                  .split(sample, countries))
    for (model, estimator), specification in model_specs.items():
        X = model_matrices[target][model]
        predicted = oof_prediction(specification, X, y, splits, countries)
        predictions[(target, model, estimator)] = predicted
        rows.append({'target': target, 'model': model, 'estimator': estimator,
                     'n_names': len(y), 'n_variables': X.shape[1], **regression_metrics(y, predicted)})
regression_results = pd.DataFrame(rows)
reference = regression_results.loc[regression_results.model.eq('country'),
                                   ['target', 'estimator', 'r2_oof', 'mae_oof']]
regression_results = regression_results.merge(reference, on=['target', 'estimator'], suffixes=('', '_country'))
regression_results['delta_r2_vs_country'] = regression_results.r2_oof - regression_results.r2_oof_country
regression_results['delta_mae_vs_country'] = regression_results.mae_oof - regression_results.mae_oof_country
regression_results = regression_results.drop(columns=['r2_oof_country', 'mae_oof_country'])
display(regression_results.loc[regression_results.target.eq('Global')].drop(columns='target')
        .round(5).set_index(['model', 'estimator']))
display(regression_results.loc[regression_results.target.ne('Global')].pivot(
    index=['model', 'estimator'], columns='target', values=['r2_oof', 'mae_oof']).round(5))

In [ ]:
best = (regression_results.loc[regression_results.target.eq('Global') & regression_results.model.ne('country')]
        .sort_values('r2_oof', ascending=False).iloc[0])
sample = samples['Global']
predicted = predictions[('Global', best['model'], best.estimator)]
fig, ax = plt.subplots(figsize=(11, 9))
prediction_plot(ax, sample.Global.to_numpy(), predicted, sample.country, '')
plt.setp([ax.xaxis.label, ax.yaxis.label], fontsize=14)
ax.tick_params(labelsize=12)
ax.legend(frameon=False, fontsize=12)
for name, x, y in zip(sample.name, predicted, sample.Global):
    ax.annotate(name, (x, y), xytext=(3, 3), textcoords='offset points', fontsize=10,
                color='dimgray', fontfamily=['Noto Sans', 'Noto Sans CJK JP'])
finish_figure(fig, 'regression_observed_vs_predicted')

In [ ]:
weight_model = (regression_results.loc[regression_results.target.eq('Global')
                                       & regression_results.model.str.contains('freq|topk_mean')]
                .sort_values('r2_oof', ascending=False).iloc[0])
model, estimator = weight_model['model'], weight_model.estimator
sample = samples['Global']
fitted = fit_model(model_specs[(model, estimator)], model_matrices['Global'][model],
                   sample.Global.to_numpy(), sample.country.to_numpy())
pipeline = fitted.best_estimator_ if isinstance(fitted, GridSearchCV) else fitted
coefficients = pipeline[-1].coef_
n = len(selected)
feature_weights = selected[['layer', 'feature_id']].copy()
feature_weights['coef_freq'] = coefficients[:n] if 'freq' in model else 0.
feature_weights['coef_topk_mean'] = (coefficients[n:2*n] if 'freq' in model and 'topk_mean' in model
                                   else coefficients[:n] if 'topk_mean' in model else 0.)
feature_weights['weight'] = np.hypot(feature_weights.coef_freq, feature_weights.coef_topk_mean)
feature_weights = feature_weights.sort_values('weight', ascending=False)
display(feature_weights.head(20).round(5).set_index(['layer', 'feature_id']))
if model.startswith('country'):
    controls = encoders['Global'].get_feature_names_out(['country'])
    display(pd.DataFrame({'control': controls, 'coefficient': coefficients[-len(controls):]}).round(5))
top = feature_weights.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh([f'L{row.layer} / {row.feature_id}' for row in top.itertuples()], top.weight)
ax.set(xlabel='Standardized coefficient weight', title=f'{model} ({estimator})')
finish_figure(fig, 'regression_feature_weights')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, target in zip(axes.flat, FACETS):
    sample = samples[target]
    row = regression_results.loc[regression_results.target.eq(target)
                                 & regression_results.model.eq('country + index')
                                 & regression_results.estimator.eq('OLS')].iloc[0]
    prediction_plot(ax, sample[target].to_numpy(), predictions[(target, 'country + index', 'OLS')],
                    sample.country, f'{target} · country + index (OLS)\nR²={row.r2_oof:.3f} · ΔR²={row.delta_r2_vs_country:+.3f}')
finish_figure(fig, 'freq_index_pooled_regressions')

In [ ]:
coefficient_rows = []
for target in TARGETS:
    sample = samples[target]
    for model in ['index', 'country + index']:
        for estimator in ['OLS', 'Ridge']:
            fitted = fit_model(model_specs[(model, estimator)], model_matrices[target][model],
                               sample[target].to_numpy(), sample.country.to_numpy())
            pipeline = fitted.best_estimator_ if isinstance(fitted, GridSearchCV) else fitted
            beta = pipeline[-1].coef_[0]
            original = beta / pipeline[0].scale_[0]
            coefficient_rows.append({'target': target, 'model': model, 'estimator': estimator,
                                     'coefficient_standardized': beta, 'coefficient_original': original,
                                     'effect_per_0.01_index': original * .01})
index_coefficients = pd.DataFrame(coefficient_rows)
display(index_coefficients.round(6).set_index(['target', 'model', 'estimator']))
coefficients = index_coefficients.loc[index_coefficients.model.eq('country + index')
                                      & index_coefficients.estimator.eq('OLS')].set_index('target').reindex(TARGETS)
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(TARGETS[::-1], coefficients['effect_per_0.01_index'].iloc[::-1])
ax.axvline(0, color='black', linewidth=.8)
ax.set(xlabel='Bias change per +0.01 frequency index', title='country + index (OLS)')
finish_figure(fig, 'freq_index_coefficients')

## Capability preservation

In [ ]:
benchmark_files = {
    'gsm8k': benchmark_summary('gsm8k', 'summary_metrics_meta_8shot_*.json', 'GSM8K_SUMMARY'),
    'culturalbench': benchmark_summary('culturalbench', 'summary_metrics_culturalbench_hard_*.json', 'CULTURALBENCH_SUMMARY'),
    'mmlu': benchmark_summary('mmlu', 'summary_metrics_mmlu_5shot_*.json', 'MMLU_SUMMARY'),
}
benchmark_frames = []
for benchmark, path in benchmark_files.items():
    records = pd.DataFrame(json.loads(path.read_text()))
    baseline_accuracy = records.loc[records.condition.eq('baseline'), 'accuracy'].iloc[0]
    frame = records[['country', 'facet', 'accuracy']].copy()
    frame.loc[records.condition.eq('baseline'), 'country'] = 'benchmark'
    frame[benchmark] = 100 * frame.pop('accuracy')
    frame[f'delta_{benchmark}'] = frame[benchmark] - 100 * baseline_accuracy
    benchmark_frames.append(frame)
capabilities = benchmark_frames[0]
for frame in benchmark_frames[1:]:
    capabilities = capabilities.merge(frame, on=['country', 'facet'], how='outer')
country_order = ['benchmark', 'Finland', *COUNTRIES, 'Greece', 'Canada']
capabilities = (capabilities.assign(
    country_order=capabilities.country.map({country: i for i, country in enumerate(country_order)}),
    facet_order=capabilities.facet.map({facet: i for i, facet in enumerate(FACETS.values())}))
    .sort_values(['country_order', 'facet_order']).drop(columns=['country_order', 'facet_order'])
    .reset_index(drop=True))
capabilities['facet'] = capabilities.facet.where(capabilities.facet.notna(), None)
benchmark_columns = list(benchmark_files)
benchmark_deltas = [f'delta_{benchmark}' for benchmark in benchmark_columns]
show_table(capabilities, benchmark_columns, benchmark_deltas, delta_unit='pp')

In [ ]:
capabilities_by_country = capabilities.iloc[1:].groupby('country', sort=False).mean(numeric_only=True).reset_index()
capabilities_by_country.insert(1, 'facet', None)
capabilities_by_country = pd.concat([capabilities.iloc[:1], capabilities_by_country], ignore_index=True)
show_table(capabilities_by_country, benchmark_columns, benchmark_deltas, delta_unit='pp')